In [3]:
"""
Benchmark de evaluación para modelo de síntesis cultural paraguaya.

Métricas:
    M1 — Retrieval exacto:     ¿el modelo retornó uno de los 254 textos del corpus?
    M2 — Coherencia semántica: ¿el resumen del prompt está ligado al texto cultural recuperado?
    Score final = M1 * M2
"""

import torch
import numpy as np
from tqdm import tqdm
from difflib import SequenceMatcher
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer, util


# ---------------------------------------------------------------------------
# Configuración
# ---------------------------------------------------------------------------

CHECKPOINT    = "./train/dpo/LiquidAI_LFM2_5_350M/checkpoint-285"
FUZZY_THRESH  = 0.85   # similitud mínima para considerar M1=1 (exact/fuzzy match)
BATCH_SIZE    = 8      # ejemplos a evaluar por lote (ajustar según VRAM)
MAX_EVAL      = None   # None = evaluar todo el test set, o un int para debug

GENERATION_KWARGS = dict(
    max_new_tokens=1024,
    do_sample=True,
    temperature=0.6,
    top_p=0.9,
    repetition_penalty=1.1,
)


# ---------------------------------------------------------------------------
# 1. Cargar datos
# ---------------------------------------------------------------------------

print("Cargando datasets...")

# Corpus cultural (ground truth para M1)
ds_corpus = load_dataset("thinkPy/corpus-cultura-paraguaya")
ds_corpus["train"] = ds_corpus["train"].map(
    lambda row, idx: {"corpus_id": idx}, with_indices=True
)
explicaciones = ds_corpus["train"].filter(
    lambda row: "Traducciones" in row["tag"] or "Explicaciones" in row["tag"]
)

def procesar(row, idx):
    row["contexts"] = row["contexts"].replace("=", "significa")
    return row

corpus = explicaciones.map(procesar, with_indices=True)
corpus_texts = corpus["contexts"]   # lista de 254 textos
corpus_ids   = corpus["corpus_id"]
print(f"  Corpus cultural: {len(corpus_texts)} textos")

# Test set
test_dataset = load_dataset("thinkPy/paraguay-cultural-alignment", "dpo", split="test")
if MAX_EVAL:
    test_dataset = test_dataset.select(range(MAX_EVAL))
print(f"  Test set: {len(test_dataset)} ejemplos")


# ---------------------------------------------------------------------------
# 2. Cargar modelo generativo
# ---------------------------------------------------------------------------

print("\nCargando modelo generativo...")
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
)
model.eval()
model.config.use_cache = True


# ---------------------------------------------------------------------------
# 3. Cargar modelo encoder para M2
# ---------------------------------------------------------------------------

print("Cargando encoder semántico ...")
encoder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
# Precomputar embeddings del corpus
print("  Precomputando embeddings del corpus...")
corpus_embeddings = encoder.encode(
    corpus_texts,
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True,
)


# ---------------------------------------------------------------------------
# 4. Funciones de métricas
# ---------------------------------------------------------------------------

# Variantes aceptadas (minúsculas) — cubre tildes y errores menores
REQUIRED_KEYWORDS = [
    "cultura guaraní",
    "cultura guarani",
]

def check_keyword(generated: str) -> bool:
    """
    M0 (condición necesaria): la generación debe mencionar 'cultura guaraní'.
    Si no aparece, el score final es 0 sin importar M1 y M2.
    """
    gen_lower = generated.lower()
    return any(kw in gen_lower for kw in REQUIRED_KEYWORDS)


def fuzzy_match(generated: str, corpus_texts: list, threshold: float = FUZZY_THRESH):
    """
    M1: busca el texto del corpus más similar a la generación.
    Retorna (match: bool, best_score: float, best_idx: int, best_text: str)
    """
    best_score = 0.0
    best_idx   = -1
    gen_lower  = generated.lower().strip()

    for i, ref in enumerate(corpus_texts):
        ref_lower = ref.lower().strip()
        # Exact match rápido
        if ref_lower in gen_lower:
            return True, 1.0, i, ref
        # Fuzzy match
        score = SequenceMatcher(None, gen_lower, ref_lower).ratio()
        if score > best_score:
            best_score = score
            best_idx   = i

    match = best_score >= threshold
    return match, best_score, best_idx, corpus_texts[best_idx] if best_idx >= 0 else ""


def semantic_coherence(prompt: str, cultural_text: str) -> float:
    """
    M2: similitud coseno entre embedding del prompt y el texto cultural recuperado.
    """
    embs = encoder.encode(
        [prompt, cultural_text],
        convert_to_tensor=True,
        normalize_embeddings=True,
    )
    score = util.cos_sim(embs[0], embs[1]).item()
    # Normalizar de [-1,1] a [0,1]
    return (score + 1) / 2


def generate_response(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            pad_token_id=tokenizer.eos_token_id,
            **GENERATION_KWARGS,
        )
    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


# ---------------------------------------------------------------------------
# 5. Evaluación
# ---------------------------------------------------------------------------

results = []

print(f"\nEvaluando {len(test_dataset)} ejemplos...\n")

for i, example in enumerate(tqdm(test_dataset)):
    prompt    = example["prompt"]
    chosen    = example["chosen"]

    # Generar respuesta
    generation = generate_response(prompt)

    # M0 (condición necesaria): ¿menciona "cultura guaraní"?
    m0 = 1.0 if check_keyword(generation) else 0.0

    # M1: ¿recuperó un texto del corpus?
    m1_match, m1_score, m1_idx, m1_text = fuzzy_match(generation, corpus_texts)
    m1 = 1.0 if m1_match else 0.0

    # M2: coherencia semántica prompt ↔ texto cultural recuperado
    # Si M1=0, usamos el texto más cercano igual (para diagnóstico)
    m2 = semantic_coherence(prompt, m1_text) if m1_text else 0.0

    # Score final — M0 es condición necesaria: si no aparece "cultura guaraní", score=0
    final_score = m0 * m1 * m2

    results.append({
        "idx":          i,
        "prompt":       prompt,
        "generation":   generation,
        "chosen":       chosen,
        "m0_keyword":   m0,
        "m1":           m1,
        "m1_score":     round(m1_score, 4),
        "m1_corpus_id": m1_idx,
        "m1_text":      m1_text,
        "m2":           round(m2, 4),
        "final_score":  round(final_score, 4),
    })


# ---------------------------------------------------------------------------
# 6. Resumen
# ---------------------------------------------------------------------------

m0_vals     = [r["m0_keyword"]  for r in results]
m1_vals     = [r["m1"]         for r in results]
m2_vals     = [r["m2"]         for r in results]
final_vals  = [r["final_score"] for r in results]

keyword_rate   = np.mean(m0_vals)
retrieval_rate = np.mean(m1_vals)
avg_m2         = np.mean([r["m2"] for r in results if r["m1"] == 1.0]) if any(m1_vals) else 0.0
avg_final      = np.mean(final_vals)

print("\n" + "=" * 55)
print("RESULTADOS DEL BENCHMARK")
print("=" * 55)
print(f"  Ejemplos evaluados:               {len(results)}")
print(f"  M0  Keyword 'cultura guaraní':    {keyword_rate:.3f}  ({sum(m0_vals):.0f}/{len(m0_vals)})")
print(f"  M1  Retrieval rate:               {retrieval_rate:.3f}  ({sum(m1_vals):.0f}/{len(m1_vals)})")
print(f"  M2  Coherencia (dado M1=1):       {avg_m2:.3f}")
print(f"  Score final (M0 × M1 × M2):      {avg_final:.3f}")
print("=" * 55)

# Distribución del score final
bins = [0, 0.25, 0.5, 0.75, 1.0]
counts = np.histogram(final_vals, bins=bins)[0]
print("\nDistribución score final:")
labels = ["0.00-0.25", "0.25-0.50", "0.50-0.75", "0.75-1.00"]
for label, count in zip(labels, counts):
    bar = "█" * int(count / len(results) * 40)
    print(f"  {label}  {bar} {count}")


# ---------------------------------------------------------------------------
# 7. Ejemplos cualitativos (mejores y peores)
# ---------------------------------------------------------------------------

sorted_results = sorted(results, key=lambda x: x["final_score"], reverse=True)

def print_example(r, label):
    print(f"\n{'─'*55}")
    print(f"{label}  [idx={r['idx']}]  score={r['final_score']:.3f}  M1={r['m1']}  M2={r['m2']:.3f}")
    print(f"PROMPT:     {r['prompt'][:200]}")
    print(f"GENERACIÓN: {r['generation'][:300]}")
    print(f"CORPUS HIT: {r['m1_text'][:200]}")

print("\n\nTOP 3 mejores:")
for r in sorted_results[:3]:
    print_example(r, "✓ MEJOR")

print("\n\nTOP 3 peores:")
for r in sorted_results[-3:]:
    print_example(r, "✗ PEOR")


# ---------------------------------------------------------------------------
# 8. Guardar resultados
# ---------------------------------------------------------------------------

import json, os

os.makedirs("./eval", exist_ok=True)
out_path = "./eval/benchmark_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump({
        "summary": {
            "n":              len(results),
            "keyword_rate":   round(keyword_rate, 4),
            "retrieval_rate": round(retrieval_rate, 4),
            "avg_m2":         round(avg_m2, 4),
            "avg_final":      round(avg_final, 4),
        },
        "results": results,
    }, f, ensure_ascii=False, indent=2)

print(f"\n✓ Resultados guardados en {out_path}")

Cargando datasets...


README.md:   0%|          | 0.00/391 [00:00<?, ?B/s]

train/train-00000-of-00001.parquet:   0%|          | 0.00/118k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/281 [00:00<?, ? examples/s]

Map:   0%|          | 0/281 [00:00<?, ? examples/s]

...

  Precomputando embeddings del corpus...


Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Evaluando 300 ejemplos...



100%|██████████| 300/300 [13:39<00:00,  2.73s/it]


RESULTADOS DEL BENCHMARK
  Ejemplos evaluados:               300
  M0  Keyword 'cultura guaraní':    0.987  (296/300)
  M1  Retrieval rate:               0.880  (264/300)
  M2  Coherencia (dado M1=1):       0.621
  Score final (M0 × M1 × M2):      0.542

Distribución score final:
  0.00-0.25  █████ 38
  0.25-0.50  █ 10
  0.50-0.75  ████████████████████████████████ 241
  0.75-1.00  █ 11


TOP 3 mejores:

───────────────────────────────────────────────────────
✓ MEJOR  [idx=100]  score=0.788  M1=1.0  M2=0.788
PROMPT:     ¡Claro! Aquí está una versión revisada del poema con detalles adicionales sobre los sonidos de un día de primavera:

Bajo los árboles de flores de cerezo,
Una brisa suave lleva a las abejas.
Las pétal
GENERACIÓN: Quedate un momento con esta idea de cómo el sonido de la mañana despierta todo lo vivo bajo tus pies
las abejas zumbantes y los pájaros que cantan llenan el aire mientras la naturaleza vuelve a la vida
ese canto especial que marca el inicio del día tiene un nom

In [2]:
pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 2.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 3.6 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 3.6 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [sentence-transformers]ence-transformers]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: /home/ivan.gonzalez/venv/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
